# Streaming Subscriber Analytics

Analysis of a streaming service's subscriber base (Netflix Userbase schema): who the subscribers are, what drives revenue, and which segments look most likely to lapse.

**Data:** by default this loads `data/raw/netflix_userbase_sample.csv`, a synthetic stand-in generated by `scripts/generate_sample_data.py` for testing. Swap in the real [Netflix Userbase Dataset](https://www.kaggle.com/datasets/arnavsmayan/netflix-userbase-dataset) from Kaggle (same column names) to analyze real data — see `README.md`.

**Note on the "lapsed" label:** this dataset has no true churn flag. `is_lapsed` is a recency proxy (last payment more than 45 days before the most recent payment in the dataset) — treat the churn-related findings below as a demonstration of the analytical workflow, not a verified churn signal.

In [1]:
from subscriber_analytics.loading import load_subscribers
from subscriber_analytics.cleaning import clean_subscribers
from subscriber_analytics import analysis, viz

DATA_PATH = "../data/raw/netflix_userbase_sample.csv"

raw = load_subscribers(DATA_PATH)
df = clean_subscribers(raw)
print(f"{len(df)} subscribers after cleaning "
      f"({df.attrs['rows_dropped_in_cleaning']} rows dropped)")
df.head()

2500 subscribers after cleaning (0 rows dropped)


,user_id,subscription_type,monthly_revenue,join_date,last_payment_date,country,age,gender,device,plan_duration,tenure_days,days_since_last_payment,is_lapsed
0,1,Premium,18.06,2024-03-21,2024-03-21,United States,54,Female,Smart TV,3 Months,0,72,True
1,2,Standard,15.36,2023-07-18,2024-05-19,Brazil,33,Female,Smart TV,3 Months,306,13,False
2,3,Premium,19.89,2024-03-03,2024-05-22,Mexico,34,Male,Smartphone,1 Month,80,10,False
3,4,Standard,15.65,2023-09-10,2023-11-22,United States,59,Male,Tablet,1 Month,73,192,True
4,5,Basic,9.82,2022-12-16,2024-05-09,United States,46,Male,Smart TV,1 Month,510,23,False


## 1. Overview

In [2]:
summary = analysis.summarize(df)
summary

{'n_subscribers': 2500,
 'total_monthly_revenue': np.float64(36020.12),
 'avg_monthly_revenue': np.float64(14.41),
 'avg_age': np.float64(38.5),
 'lapsed_rate': np.float64(0.1688),
 'reference_date': Timestamp('2024-06-01 00:00:00'),
 'earliest_join_date': Timestamp('2022-06-03 00:00:00'),
 'latest_join_date': Timestamp('2024-05-02 00:00:00')}

## 2. Revenue
Where does monthly revenue come from — by plan, country, and device?

In [3]:
revenue_by_plan = analysis.revenue_by(df, "subscription_type")
viz.revenue_by_plan_chart(revenue_by_plan, "subscription_type").show()
revenue_by_plan

,subscription_type,subscribers,avg_revenue,total_revenue
0,Standard,968,14.98,14495.82
1,Premium,646,18.98,12263.59
2,Basic,886,10.45,9260.71


In [4]:
revenue_by_country = analysis.revenue_by(df, "country")
viz.revenue_by_plan_chart(revenue_by_country, "country").show()

In [5]:
device_segments = analysis.segment_counts(df, "device")
viz.segment_share_pie(device_segments, "device").show()

## 3. Growth over time
Signups by join month — is the subscriber base growing, flat, or shrinking?

In [6]:
viz.signups_over_time_chart(df).show()

## 4. Who's at risk of lapsing?
Age distribution and lapsed rate by plan, plus hypothesis tests on whether these differences are statistically meaningful.

In [7]:
viz.age_distribution_histogram(df).show()

In [8]:
viz.lapsed_rate_by_plan_chart(df).show()

In [9]:
print(analysis.chi_square_association(df, "subscription_type", "device"))
print(analysis.anova_revenue_by_plan(df))
print(analysis.ttest_age_by_lapsed_status(df))

{'test': 'chi-square', 'columns': ('subscription_type', 'device'), 'chi2': np.float64(2.181), 'p_value': np.float64(0.9023), 'dof': 6, 'significant_at_0.05': False}
{'test': 'one-way ANOVA', 'target': 'monthly_revenue', 'grouped_by': 'subscription_type', 'f_stat': np.float64(29270.881), 'p_value': np.float64(0.0), 'significant_at_0.05': True}
{'test': "Welch's t-test", 'target': 'age', 'grouped_by': 'is_lapsed', 'lapsed_mean_age': np.float64(39.0), 'active_mean_age': np.float64(38.4), 't_stat': np.float64(0.96), 'p_value': np.float64(0.3375), 'significant_at_0.05': False}


## 5. Churn-proxy model
A logistic regression predicting the `is_lapsed` proxy from age, tenure, revenue, plan, device, and gender. This is a workflow demonstration — accuracy on a proxy label isn't evidence of a deployable churn model.

In [10]:
result = analysis.fit_churn_proxy_model(df)
print(f"accuracy={result.accuracy}  roc_auc={result.roc_auc}  "
      f"n_train={result.n_train}  n_test={result.n_test}")
result.confusion_matrix

accuracy=0.832  roc_auc=0.5722  n_train=1875  n_test=625


array([[520,   0],
       [105,   0]])

In [11]:
viz.churn_model_coefficients_chart(result.coefficients).show()

## Takeaways

- Fill this in once you're looking at the real dataset — the sample data's patterns are synthetic and won't reflect genuine subscriber behavior.
- Swap `DATA_PATH` above to the real CSV and re-run top to bottom; every function here is dataset-agnostic as long as the column schema matches.